In [ ]:
query = 'List all teams that have never won the FIFA world cup but have reached the top 3'

In [ ]:
import os
from google.colab import userdata


os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('aws_access_key')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('aws_secret_key')

In [ ]:
from llama_index.llms.bedrock_converse import BedrockConverse

llm = BedrockConverse(
    model='anthropic.claude-3-haiku-20240307-v1:0',
    region_name='us-east-1',
    temperature=1
)
response = llm.complete(query)

In [ ]:
print(response)

Here is a list of teams that have never won the FIFA World Cup but have reached the top 3 (finished in the top 3):

1. Netherlands - Runners-up in 1974, 1978, and 2010
2. Croatia - Finished third in 1998
3. Hungary - Finished second in 1938 and 1954
4. Sweden - Finished second in 1958
5. Czechoslovakia - Finished second in 1934 and 1962
6. Belgium - Finished third in 1986
7. Poland - Finished third in 1974 and 1982
8. England - Finished fourth in 1966
9. Austria - Finished third in 1954
10. Uruguay - Finished second in 1930, 1950, and 1970
11. France - Finished second in 2006 and third in 1958 and 1986

Note that this list only includes teams that have never won the World Cup. Many of these teams have won other major international tournaments, such as the European Championship or the Copa América.


In [ ]:
wiki_pages = [
    f'https://en.wikipedia.org/wiki/{year}_FIFA_World_Cup'
    for year in range(1930, 2025, 4)
]

In [ ]:
from llama_index.readers.wikipedia import WikipediaReader

In [ ]:
wikis = WikipediaReader().load_data(wiki_pages)

In [ ]:
from llama_index.core.node_parser.text.sentence import SentenceSplitter

In [ ]:
wiki_chunks = SentenceSplitter(chunk_size=100, chunk_overlap=20).get_nodes_from_documents(wikis)
len(wiki_chunks)

2112

In [ ]:
from llama_index.core import Settings
from llama_index.embeddings.bedrock import BedrockEmbedding


embed_model = BedrockEmbedding(
    model_name='cohere.embed-english-v3',
    region_name='us-east-1',
)

Settings.llm = llm
Settings.embed_model = embed_model

In [ ]:
from llama_index.core import VectorStoreIndex


engine = VectorStoreIndex.from_documents(wiki_chunks, show_progress=True).as_query_engine()

In [ ]:
response = engine.query(query)

In [ ]:
response.response

'Based on the information provided in the context, the team that has never won the FIFA World Cup but has reached the top 3 is Argentina. The context states that Argentina was the runner-up in the first ever World Cup in 1930, indicating they have reached the top 3 of the competition without ever winning the title.'

In [ ]:
evidence = '\n'.join([node.text for node in response.source_nodes])
print(evidence)

However, after being the runner-up for the first ever World Cup in 1930, Argentina somewhat fell behind Brazil on the global stage, where the latter went on to win three World Cups in 1958, 1962, and 1970, while their figurehead player Pelé became the face of international football.


In [ ]:
follow_up_prompt = """
Given the following question, think of any piece of information you might need in order to answer it.
More specifically, the only source of information is wikipedia, so if you need any information from there, respond with a follow up question.
Don't provide any thoughts or reasoning - simply respond with a question.
If you don't need more information to answer the question, just respond with DONE.
Other than follow up question (or DONE), respond with nothing.

Question: {query}

Respond with a follow up question or with DONE.
"""

In [ ]:
llm.complete(follow_up_prompt.format(query=query)).text

'What are the criteria for reaching the "top 3" in the FIFA World Cup?'

In [ ]:
from tqdm import trange

In [ ]:
def create_cot(engine):
  num_of_iterations_cot = 5
  follow_up_prompt = """
  Given the following question, think of any piece of information you might need in order to answer it.
  More specifically, the only source of information is wikipedia, so if you need any information from there, respond with a follow up question.
  Don't provide any thoughts or reasoning - simply respond with a question.
  If you don't need more information to answer the question, just respond with DONE.
  Other than follow up question (or DONE), respond with nothing.

  Question: {query}

  Past follow up questions, evidences and answers:
  {q_and_a}

  Respond with a different follow up question or with DONE. Do not repeat the previous follow up questions or the original question.
  """
  q_and_a = ''

  for _ in trange(num_of_iterations_cot):
    follow_up_question = llm.complete(
        follow_up_prompt.format(
            query=query,
            q_and_a=q_and_a
        )
    ).text
    print(follow_up_question)
    if follow_up_question.strip().lower() == 'done':
      break
    response = engine.query(follow_up_question)
    evidences = '\n'.join([node.text for node in response.source_nodes])
    answer = response.response

    q_and_a += f"""
    - Followup question: {follow_up_question}
    - Evidences: {evidences}
    - Answer: {answer}
    """

  return q_and_a

In [ ]:
q_and_a = create_cot()

In [ ]:
prompt = """
Given the following question and follow up questions and their evidences and answer,
I want to you to answer the question.
Here is the question you should answer:
{query}

Here are the follow up question, their evidences and their answers:
{q_and_a}

Respond with an answer, and nothing more.
"""

llm.complete(
    prompt.format(
        query=query,
        q_and_a=q_and_a
    )
).text

'Hungary, Netherlands, Croatia.'

In [ ]:
no_overlap_wiki_chunks = SentenceSplitter(
    chunk_size=100,
    chunk_overlap=0
).get_nodes_from_documents(wikis)
no_overlap_engine = VectorStoreIndex.from_documents(
    no_overlap_wiki_chunks,
    show_progress=True
).as_query_engine()

In [ ]:
from tqdm import tqdm

engines = [
    engine,
    no_overlap_engine,
    engine,
]
cots = [create_cot(engine) for engine in tqdm(engines)]

  0%|          | 0/5 [00:00<?, ?it/s]

What are the teams that have reached the top 3 in the FIFA World Cup?



 20%|██        | 1/5 [00:02<00:09,  2.49s/it]

What are the FIFA World Cup third place finishes for teams that have never won the tournament?



 40%|████      | 2/5 [00:04<00:05,  2.00s/it]

...


 60%|██████    | 3/5 [00:31<00:20, 10.30s/it]

What countries have finished in the top 3 of the FIFA World Cup tournament without ever winning the championship?



 80%|████████  | 4/5 [00:48<00:12, 12.80s/it]

What are the specific FIFA World Cup tournaments that are considered in determining which teams have "never won the FIFA World Cup but have reached the top 3"?



100%|██████████| 3/3 [02:04<00:00, 41.57s/it]


In [ ]:
print('\n\n'.join(cots))


    - Followup question: What are the teams that have reached the top 3 in the FIFA World Cup?
    - Evidences: Moreover, throughout football's history, both nations have had players who, at one point or another, were considered to be the very best in the world, such as Alfredo Di Stéfano, Diego Maradona, and Lionel Messi from Argentina, and Pelé, Ronaldo, and Ronaldinho from Brazil.
In a head-to-head comparison of senior titles, Brazil has won five FIFA World Cups, while Argentina has won three.
    - Answer: According to the context provided, the teams that have reached the top 3 in the FIFA World Cup are Brazil and Argentina. The context states that Brazil has won five FIFA World Cups, while Argentina has won three. This indicates that these two nations have been among the top performing teams in the FIFA World Cup over the years.
    
    - Followup question: What are the FIFA World Cup third place finishes for teams that have never won the tournament?
    - Evidences: Nevertheles

In [ ]:
prompt = """
Given the following question and follow up questions and their evidences and answer,
I want to you to answer the question.
Here is the question you should answer:
{query}

Here are the follow up question, their evidences and their answers:
{q_and_a}

Respond with an answer, and nothing more.
"""

llm.complete(
    prompt.format(
        query=query,
        q_and_a='\n\n'.join(cots)
    )
).text

'Brazil and Argentina.'